# Probability distributions: calculations, visualizations, and applications

A hands-on introduction to eight probability distributions using **SciPy**, **NumPy**, **Matplotlib**, and this repository's `probability_functions.py` module.

**Learning goals:** choose a distribution for a scenario, interpret its parameters, distinguish probability from probability density, calculate cumulative and tail probabilities, and visualize the results.

All examples below are **hypothetical illustrations**, not measurements from a real organization. The notebook uses the corrected library in the parent repository; it is intended to be run from the repository root (or by following the setup instructions below).

## 1. Setup and probability conventions

Install dependencies from the repository root:

```bash
python -m pip install -r requirements.txt jupyterlab
jupyter lab
```

Open `notebooks/probability_distributions.ipynb`. This notebook locates the repository module when started from either the repository root or the `notebooks/` directory.

**Discrete distributions:** a probability mass function (PMF) gives `P(X = k)`. **Continuous distributions:** a probability density function (PDF) is *not* the probability at a point: `P(X = x) = 0`, and probabilities are areas under the density curve.

Our `probability(spec, x, comparison)` helper supports `"="`, `"<="`, and `">="`. The `">="` comparison includes the boundary for discrete outcomes.

In [ ]:
from pathlib import Path
import sys

# Works when the notebook's working directory is the repository root or notebooks/.
repo_root = Path.cwd()
if not (repo_root / "probability_functions.py").exists():
    repo_root = repo_root.parent
if not (repo_root / "probability_functions.py").exists():
    raise FileNotFoundError("Open this notebook from the repository root or notebooks/ directory.")
sys.path.insert(0, str(repo_root))

import matplotlib.pyplot as plt
import numpy as np

from probability_functions import (
    make_distribution,
    probability,
    density,
    plot_distribution,
    summary_statistics,
)

def show_example(name, parameters, x, comparisons=("=", "<=", ">=")):
    """Print probabilities and summary moments for one scenario."""
    spec = make_distribution(name, **parameters)
    print(f"{spec.name} ({spec.description})")
    for operator in comparisons:
        print(f"  P(X {operator} {x}) = {probability(spec, x, operator):.6f}")
    if not spec.discrete:
        print(f"  PDF at x={x} = {density(spec, x):.6f} (a density, not a probability)")
    for moment, value in summary_statistics(spec).items():
        print(f"  {moment}: {value:.4f}")
    return spec

print("Notebook environment ready.")

## 2. Binomial distribution

**Question:** A hypothetical inspection process has a 3% defect probability per unit. What is the probability that exactly two of the next 20 inspected units are defective?

Assumptions: 20 **independent** Bernoulli trials, each with the same defect probability `p = 0.03`.

\[
X \sim \operatorname{Binomial}(n,p), \qquad
P(X=k)=\binom{n}{k}p^k(1-p)^{n-k}.
\]

Useful for counts of successes in a fixed number of independent trials; not for samples without replacement from a small finite batch.

In [ ]:
binomial_spec = show_example("binomial", {"n": 20, "p": 0.03}, x=2)
plot_distribution(binomial_spec)
print("Interpretation: 'P(X >= 2)' is the probability of at least two defective units.")

## 3. Poisson distribution

**Question:** A hypothetical service desk receives an average of 4 requests per hour. What is the probability of at least 6 requests in the next hour?

Assume a constant arrival rate and approximately independent arrivals.

\[
X \sim \operatorname{Poisson}(\lambda), \qquad
P(X=k)=e^{-\lambda}\frac{\lambda^k}{k!}.
\]

Here `lam` is the **expected number of arrivals over the selected time window**. Change it proportionally when changing the time window under a constant-rate assumption.

In [ ]:
poisson_spec = show_example("poisson", {"lam": 4}, x=6)
plot_distribution(poisson_spec)
print("At least six arrivals:", f"{probability(poisson_spec, 6, '>='):.2%}")

## 4. Normal distribution

**Question:** Suppose a manufactured part's length is approximately normal with mean 500 mm and standard deviation 12 mm. What proportion is expected to measure 515 mm or less?

\[
X \sim N(\mu,\sigma^2), \qquad
f(x)=\frac{1}{\sigma\sqrt{2\pi}}\exp\!\left(-\frac{(x-\mu)^2}{2\sigma^2}\right).
\]

For a continuous measurement, the probability of **exactly** 515 mm is zero. `density()` reports the **PDF height**, while `probability(..., "<=")` calculates the CDF.

In [ ]:
normal_spec = show_example("normal", {"mu": 500, "sigma": 12}, x=515)
plot_distribution(normal_spec)
print("Z-score for 515 mm:", (515 - 500) / 12)

## 5. Exponential distribution

**Question:** Under a constant hazard/rate model, the average wait for a hypothetical service event is 8 minutes. What is the probability the wait is 10 minutes or less?

\[
X\sim\operatorname{Exponential}(\lambda),\qquad
f(x)=\lambda e^{-\lambda x},\; x\ge 0,\qquad E[X]=1/\lambda.
\]

**Important:** The toolkit takes `lam` as a **rate**. For an 8-minute mean, `lam = 1/8` per minute. In SciPy, this corresponds to `stats.expon(scale=8)`.

In [ ]:
exponential_spec = show_example("exponential", {"lam": 1 / 8}, x=10)
plot_distribution(exponential_spec)
print("Probability wait exceeds 10 minutes:",
      f"{probability(exponential_spec, 10, '>='):.2%}")

## 6. Geometric distribution

**Question:** With independent attempts and a 20% chance of success per attempt, what is the probability that the first success occurs on the fourth attempt?

\[
X\sim\operatorname{Geometric}(p),\qquad
P(X=k)=(1-p)^{k-1}p,\; k=1,2,\ldots
\]

Here the random variable counts **trials through the first success**, not failures before the first success.

In [ ]:
geometric_spec = show_example("geometric", {"p": 0.2}, x=4)
plot_distribution(geometric_spec)

## 7. Hypergeometric distribution

**Question:** A hypothetical batch contains 50 items, of which 8 are defective. If we inspect 6 distinct items **without replacement**, what is the probability of finding exactly 2 defectives?

\[
X\sim\operatorname{Hypergeometric}(N,K,n),\qquad
P(X=k)=\frac{\binom{K}{k}\binom{N-K}{n-k}}{\binom{N}{n}}.
\]

Use the hypergeometric rather than binomial model when the population is finite, the sample is taken without replacement, and the exact count of successes in the population is known.

In [ ]:
hypergeom_spec = show_example("hypergeometric", {"N": 50, "K": 8, "n": 6}, x=2)
plot_distribution(hypergeom_spec)

# Compare with a binomial approximation using p = K/N.
approx = make_distribution("binomial", n=6, p=8 / 50)
print("Hypergeometric P(X = 2):", round(probability(hypergeom_spec, 2, "="), 6))
print("Binomial approximation:", round(probability(approx, 2, "="), 6))

## 8. Negative binomial distribution

**Question:** Suppose an independent attempt succeeds with probability 40%. What is the probability that there are exactly 4 failures **before** the third success?

\[
X\sim\operatorname{NegBinomial}(r,p),\qquad
P(X=k)=\binom{k+r-1}{k}p^r(1-p)^k.
\]

**Convention:** This implementation (matching SciPy's `nbinom`) counts **failures before `r` successes**, not the total number of attempts. Total attempts would be `X + r`.

In [ ]:
negative_binomial_spec = show_example("negative_binomial", {"r": 3, "p": 0.4}, x=4)
plot_distribution(negative_binomial_spec)
print("Exactly 7 attempts for the third success:",
      f"{probability(negative_binomial_spec, 4, '='):.2%}")

## 9. Uniform distribution

**Question:** A hypothetical arrival time is equally likely anywhere between 2 PM and 6 PM. What is the probability it falls at or before 4 PM?

Represent the time numerically, with `a = 2`, `b = 6`, and `x = 4`.

\[
X\sim\operatorname{Uniform}(a,b),\qquad
f(x)=\frac{1}{b-a}\quad\text{for }a\le x\le b.
\]

The density is `1/(b-a)`, not `1/b`. The PDF's value may be above 1 for sufficiently narrow intervals; only the **integral** of the PDF must equal 1.

In [ ]:
uniform_spec = show_example("uniform", {"a": 2, "b": 6}, x=4)
plot_distribution(uniform_spec)
assert np.isclose(probability(uniform_spec, 4, "<="), 0.5)

## 10. Quick comparison and numerical checks

The following checks highlight four easy-to-confuse ideas:

1. A continuous random variable has zero probability at an exact point.
2. Its PDF height is *not* a point probability.
3. `P(X >= k)` includes the point mass at `k` for discrete distributions.
4. A simulated result can approximate an analytic probability, but sampling variability remains.

In [ ]:
assert probability(normal_spec, 500, "=") == 0
assert np.isclose(density(uniform_spec, 4), 0.25)
assert np.isclose(
    probability(binomial_spec, 2, ">="),
    1 - probability(binomial_spec, 1, "<="),
)

rng = np.random.default_rng(42)
samples = rng.binomial(n=20, p=0.03, size=100_000)
simulation = np.mean(samples >= 2)
analytic = probability(binomial_spec, 2, ">=")
print(f"Simulated P(X >= 2): {simulation:.4f}")
print(f"Analytic P(X >= 2):  {analytic:.4f}")
print(f"Absolute error:      {abs(simulation - analytic):.4f}")

## 11. Choosing a distribution

| Question | Typical model | Key assumption |
|---|---|---|
| Successes in `n` independent trials | Binomial | Fixed `n`, identical success probability |
| Events in a fixed interval | Poisson | Constant-rate independent arrivals |
| Symmetric continuous variation | Normal | Approximate bell-shaped measurements |
| Waiting time to an event | Exponential | Constant event rate / memoryless process |
| Attempts until first success | Geometric | Independent repeated trials |
| Successes sampled without replacement | Hypergeometric | Known finite population |
| Failures before `r` successes | Negative binomial | Independent repeated trials |
| Equal likelihood across an interval | Uniform | Constant density on a bounded interval |

### Further exploration

Try varying the parameters, then compare how the distribution's shape, tail probabilities, mean, and variance change. For more rigorous applications, check whether the modeling assumptions match the actual data before selecting a distribution.

**Sources of formulas:** standard probability definitions implemented by SciPy; this notebook intentionally uses hypothetical examples rather than presenting simulated values as observed business data.